 # 2.08b — Final Linear Regression Training on All Data

 In 2.04b I trained the Linear regressors on 2019+2021 and held 2026 out for evaluation.
 Here I refit them on the full 2019 + 2021 + 2026 dataset. The RMSE, MAE, and
 prediction-interval coverage numbers from 2.04b are the honest ones and do not change.

 There is one thing about Linear that does not apply to LightGBM or Logistic: the
 prediction intervals. The 95% PI offsets were calibrated on the 2026 holdout residuals
 in 2.04b, meaning they reflect how wrong the model actually was on data it had never
 seen. I cannot recompute them from full-data residuals here because those would be
 in-sample and far too narrow, which would give users a false sense of certainty. So
 before fitting anything, I read the offsets back out of the 2.04b artifacts and store
 them. After each new fit, I attach the same offsets unchanged.

 This means 2.04b needs to have been run before this notebook. The cell that loads the
 offsets will raise an error if those artifacts are missing.

In [ ]:
import sys
import time
import warnings
from pathlib import Path

import joblib
import numpy as np
from sklearn.linear_model import LinearRegression
from sklearn.pipeline import Pipeline

warnings.filterwarnings("ignore", category=UserWarning)

sys.path.insert(0, str(Path().resolve().parent))
from model_training.feature_prep import (
    TARGET_REGRESSION,
    TRAINING_YEARS,
    build_preprocessor,
    get_X_y,
    load_training_data,
)

 ## Setup

 Same horizons as 2.04b. MODELS_DIR is where 2.04b wrote its artifacts and where the
 new full-data artifacts will overwrite them.

In [ ]:
HORIZONS = [60, 180, 360, 720, 1440, 2880]
HORIZON_LABELS = {60: "1hr", 180: "3hr", 360: "6hr",
                  720: "12hr", 1440: "24hr", 2880: "multi-day"}

FULL_YEARS  = TRAINING_YEARS   # [2019, 2021, 2026], no holdout withheld
USE_FLOAT32 = True

MODELS_DIR = Path("../models")
MODELS_DIR.mkdir(parents=True, exist_ok=True)

print(f"Horizons: {[HORIZON_LABELS[h] for h in HORIZONS]}")
print(f"Training on: {FULL_YEARS}")
print(f"Artifacts -> {MODELS_DIR.resolve()}")

Horizons: ['1hr', '3hr', '6hr', '12hr', '24hr', 'multi-day']
Training on: [2019, 2021, 2026]
Artifacts -> C:\Users\clark\Desktop\citibike\models


 ## Read the prediction-interval offsets from 2.04b

 I want to load all 6 offsets before touching any production file. If something is
 missing the notebook stops here rather than halfway through overwriting artifacts.
 The offsets are the 2.5th and 97.5th percentiles of the 2026 holdout residuals, so
 they are grounded in how the model actually performed on unseen data. Nothing about
 those numbers changes when I refit on more rows.

In [ ]:
pi_offsets = {}
for h in HORIZONS:
    path = MODELS_DIR / f"linear_regression_{h}min.joblib"
    if not path.exists():
        raise FileNotFoundError(
            f"{path} not found. Run 2.04b first so the holdout-calibrated "
            f"PI offsets exist before this notebook overwrites them."
        )
    artifact = joblib.load(path)
    assert abs(artifact["pi_level"] - 0.95) < 1e-9, \
        f"unexpected pi_level in {path.name}: {artifact['pi_level']}"
    pi_offsets[h] = (artifact["pi_lower_offset"], artifact["pi_upper_offset"])

print("PI offsets recovered from 2.04b artifacts:")
print(f'{"horizon":>10}  {"lower":>10}  {"upper":>10}')
for h in HORIZONS:
    lo, hi = pi_offsets[h]
    print(f"{HORIZON_LABELS[h]:>10}  {lo:>10.4f}  {hi:>10.4f}")

PI offsets recovered from 2.04b artifacts:
   horizon       lower       upper
       1hr     -6.6183      6.7777
       3hr    -11.7728     12.4063
       6hr    -15.3287     15.9193
      12hr    -15.4802     16.6043
      24hr    -15.0510     16.0884
 multi-day    -16.9103     18.3982


 ## Train all 6 horizons

 One horizon at a time. Each fit loads about 14M rows through the linear pipeline
 (impute, scale, one-hot encode), runs OLS, then saves the artifact as the same dict
 shape 2.04b produced. The serving layer calls pipeline.predict(X) for the point
 estimate and adds the two offsets to get the band, clipping the lower bound at 0.

In [ ]:
for h in HORIZONS:
    label = HORIZON_LABELS[h]
    t0 = time.time()

    df = load_training_data(h, years=FULL_YEARS)
    X, y = get_X_y(df, TARGET_REGRESSION)
    if USE_FLOAT32:
        num_cols = X.select_dtypes(include=["float64", "float32"]).columns
        X[num_cols] = X[num_cols].astype("float32")
        y = y.astype("float32")
    span = f"{df['timestamp'].min().date()} -> {df['timestamp'].max().date()}"
    del df

    print(f"  {label:<10} {len(X):>11,} rows  ({span})  fitting ...", end=" ", flush=True)

    pipe = Pipeline([
        ("pre",   build_preprocessor("linear")),
        ("model", LinearRegression()),
    ])
    pipe.fit(X, y)

    pi_lower, pi_upper = pi_offsets[h]
    artifact = {
        "pipeline":        pipe,
        "pi_lower_offset": pi_lower,
        "pi_upper_offset": pi_upper,
        "pi_level":        0.95,
    }
    out = MODELS_DIR / f"linear_regression_{h}min.joblib"
    joblib.dump(artifact, out)
    print(f"saved -> {out.name}   PI [{pi_lower:+.2f}, {pi_upper:+.2f}]   ({time.time() - t0:.0f}s)")
    del X, y, pipe, artifact

  1hr         14,674,158 rows  (2019-01-01 -> 2026-06-18)  fitting ... saved -> linear_regression_60min.joblib   PI [-6.62, +6.78]   (557s)
  3hr         14,355,238 rows  (2019-01-01 -> 2026-06-17)  fitting ... saved -> linear_regression_180min.joblib   PI [-11.77, +12.41]   (492s)
  6hr         14,007,554 rows  (2019-01-01 -> 2026-06-17)  fitting ... saved -> linear_regression_360min.joblib   PI [-15.33, +15.92]   (490s)
  12hr        13,722,578 rows  (2019-01-01 -> 2026-06-17)  fitting ... saved -> linear_regression_720min.joblib   PI [-15.48, +16.60]   (515s)
  24hr        14,680,390 rows  (2019-01-01 -> 2026-06-17)  fitting ... saved -> linear_regression_1440min.joblib   PI [-15.05, +16.09]   (527s)
  multi-day   14,476,193 rows  (2019-01-01 -> 2026-06-16)  fitting ... saved -> linear_regression_2880min.joblib   PI [-16.91, +18.40]   (520s)


 ## Summary

 A quick printout confirming the offsets that ended up in each artifact, so I can
 verify at a glance that they match what 2.04b reported.

In [ ]:
print("\nLinear artifacts written:")
for h in HORIZONS:
    p = MODELS_DIR / f"linear_regression_{h}min.joblib"
    lo, hi = pi_offsets[h]
    print(f"  {HORIZON_LABELS[h]:<10} {p.name}   PI [{lo:+.2f}, {hi:+.2f}]")


Linear artifacts written:
  1hr        linear_regression_60min.joblib   PI [-6.62, +6.78]
  3hr        linear_regression_180min.joblib   PI [-11.77, +12.41]
  6hr        linear_regression_360min.joblib   PI [-15.33, +15.92]
  12hr       linear_regression_720min.joblib   PI [-15.48, +16.60]
  24hr       linear_regression_1440min.joblib   PI [-15.05, +16.09]
  multi-day  linear_regression_2880min.joblib   PI [-16.91, +18.40]


 ## Conclusion

 Six Linear regressors are now trained on the full 2019 + 2021 + 2026 dataset and saved
 to models/ with the holdout-calibrated prediction intervals attached. The intervals are
 identical to the ones reported in 2.04b because I read them back from those artifacts
 rather than recomputing them here. In-sample residuals would make the band artificially
 narrow, so this is the right call. Run 2.08c next for the Logistic classifiers.